# 13 · The chosen model on HEO, PhiSat-2 and Sentinel-2

The model written down in 12 [9] (`choice.json`, before the test was opened), on:

- every HEO scene of the test, validation and search splits at 2.5 m (on board the pixels are about 2.8 m);
- the PhiSat-2 test scenes with the most labelled fresh burn (`N_PHISAT2`);
- the Sentinel-2 test chips with the most labelled burn (`N_S2`).

For each scene: the image (NIR, red, green), the ground truth, the model's class map, and its fresh-burn errors (found in yellow, missed in cyan, false alarms in magenta) with IoU, precision, recall and hectares; when the scene has labelled fresh burn, the same four panels zoomed on it, at full resolution. The scenes are picked by role and by how much labelled burn they have, not by how well the model does on them. The numbers are counted as in 12: on the scene, every pixel once, over labelled pixels; the input is prepared as on board.

In [ ]:
# [1] setup
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")                # the notebooks run from the repository root
import os, sys, json, gc, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
import torch
from IPython.display import Image, display
for p in (os.path.join(os.getcwd(), "pyqnas", "src"), os.getcwd()):
    if p not in sys.path:
        sys.path.insert(0, p)
import pynas
from pynas import config, final, report, runner
from pynas.train import data as D
from pynas.train.metrics import SCENE_MIN_PX

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
cfg = config.load(Path("configs/burnscar.ini"))
spec = D.DataSpec.from_config(cfg)
dcfg = json.load(open("data/tables/dataset_config.json"))
SIZE = dcfg["size"]
POOL, BASE = Path(spec.pool), Path(cfg.get("Run", "out"))
R2 = BASE / "final_r2"
FQ, Q2F = R2 / "finalists" / "runs" / "queue.json", R2 / "runs" / "queue.json"
OUT = BASE / "chosen_review"
OUT.mkdir(parents=True, exist_ok=True)
HEO_ROLES = ["test", "val", "search"]   # HEO scenes at 2.5 m, in this order
N_PHISAT2, N_S2 = 4, 2
CHOICE_FILE = BASE / "review_finalists" / "choice.json"
assert CHOICE_FILE.exists(), "write the choice in 12 [9] first"
choice = json.load(open(CHOICE_FILE))
dev = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"PyNAS {pynas.__version__}; device {dev}; choice: {choice['model']} ({choice['label']}, seed {choice['seed']}), "
      f"written {choice['time']}: {choice['reason']}")

In [ ]:
# [2] the model
label, seed = choice["label"], int(float(choice["seed"]))
q = FQ if str(label).startswith("finalist") else Q2F
res = final.results(q)
run = res[(res.label == label) & (res.seed == seed)].run.iloc[0]
net = runner.load_result(Path(runner.read_queue(q)["out"]) / run, dev)
NAME = choice["model"]
models = {NAME: (net, "fp16")}
print(f"{NAME}: {choice['spec']}, {float(choice['ir_mb']):.2f} MB, run {run}")

In [ ]:
# [3] the scenes: HEO at 2.5 m by role, PhiSat-2 and Sentinel-2 test scenes with the most labelled burn
ORDER = {r: i for i, r in enumerate(HEO_ROLES)}
PH = next(n for n in ("phisat2_all_g4.75", "phisat2_6_g4.75", "phisat2_g4.75") if (POOL / "review" / f"{n}_windows.csv").exists())
PICK = []
w = pd.read_csv(POOL / "review" / "heo_g2.5_windows.csv")
k = w[w.role.isin(HEO_ROLES)].groupby("scene").role.first().reset_index()
for r in k.assign(o=k.role.map(ORDER)).sort_values(["o", "scene"]).itertuples():
    PICK.append(("review/heo_g2.5", "HEO", str(r.scene), r.role))
w = pd.read_csv(POOL / "review" / f"{PH}_windows.csv")
top = w[w.role == "test"].groupby("scene").fresh.sum().sort_values(ascending=False).index[:N_PHISAT2]
PICK += [(f"review/{PH}", "PhiSat-2 (sim.)", str(sc), "test") for sc in top]
s2 = next(n for n in spec.test if n.startswith("s2"))
s = D.open_set(spec, s2, teacher=False)
y = report.labels_of(s, spec)
burn = [spec.target] + ([list(spec.class_names).index("old")] if "old" in spec.class_names else [])
per = pd.Series(np.isin(y, burn).sum((1, 2)), index=s["windows"].scene.astype(str)).groupby(level=0).sum()
PICK += [(s2, "Sentinel-2", sc, "test") for sc in per.sort_values(ascending=False).index[:N_S2]]
del s, y
pd.DataFrame(PICK, columns=["set", "sensor", "scene", "role"])

In [ ]:
# [4] one scene: numbers and its page (whole scene, and the zoom on the labelled fresh burn)
NAMES = list(spec.class_names)
FRESH = spec.target
OLD = NAMES.index("old") if "old" in NAMES else None
CMAP = ListedColormap([report.CLASS_COLOURS.get(n, "#999999") for n in NAMES])
PANEL, MAXPX, ZOOM_MIN, ZOOM_MAX = 3.6, 1100, 192, 1000


def ha(v):
    return "–" if not np.isfinite(v) else (f"{v:,.0f}" if v >= 100 else f"{v:.1f}" if v >= 10 else f"{v:.2f}")


def f2(v):
    return "–" if not np.isfinite(v) else f"{v:.2f}"


def stretch(img):
    lo, hi = np.nanpercentile(img, 2, (0, 1)), np.nanpercentile(img, 98, (0, 1))
    return np.clip((img - lo) / np.maximum(hi - lo, 1e-6), 0, 1)


def paste_box(canvas, tiles, r, c, r0, c0):
    H, W = canvas.shape[:2]
    for t, i, j in zip(tiles, r - r0, c - c0):
        i0, j0, i1, j1 = max(i, 0), max(j, 0), min(i + t.shape[0], H), min(j + t.shape[1], W)
        if i1 > i0 and j1 > j0:
            canvas[i0:i1, j0:j1] = t[i0 - i:i1 - i, j0 - j:j1 - j]
    return canvas


def errors(p, lab, grey):
    show = np.repeat(grey, 3, -1).copy()
    show[(p == FRESH) & (lab == FRESH)] = report.HIT
    show[(p == FRESH) & (lab != FRESH) & (lab >= 0)] = report.FALSE_ALARM
    show[(p != FRESH) & (lab == FRESH)] = report.MISS
    return show


def panels(axes, img, lab, pm, gaps):
    grey = np.where(gaps[..., None], 1.0, 0.35 + 0.5 * np.nan_to_num(img).mean(-1, keepdims=True))
    axes[0].imshow(np.where(gaps[..., None], 1.0, np.nan_to_num(img)))
    axes[1].imshow(np.ma.masked_less(lab, 0), cmap=CMAP, vmin=-0.5, vmax=len(NAMES) - 0.5, interpolation="nearest")
    axes[2].imshow(np.ma.masked_where(gaps | (pm < 0), pm), cmap=CMAP, vmin=-0.5, vmax=len(NAMES) - 0.5,
                   interpolation="nearest")
    axes[3].imshow(errors(pm, lab, grey), interpolation="nearest")


def scene_page(s, pred, sc, g, role_, sensor, rgb=(6, 2, 1)):
    win = s["windows"]
    idx = np.arange(len(win))
    y = report.labels_of(s, spec)
    px_ha = g * g / 1e4
    r1, c1, h1, w1, _ = report._mosaic(idx, win, SIZE, 1, None)
    lab1 = report._paste(np.full((h1, w1), -1, np.int16), y, r1, c1, 1)
    valid, gt = lab1 >= 0, lab1 == FRESH
    pf = (report._paste(np.full((h1, w1), 255, np.uint8), pred, r1, c1, 1) == FRESH) & valid
    tp, fp, fn = int((pf & gt).sum()), int((pf & ~gt).sum()), int((gt & ~pf).sum())
    row = dict(sensor=sensor, role=role_, scene=sc, gsd=g, labelled_ha=int(gt.sum()) * px_ha,
               old_ha=int((lab1 == OLD).sum()) * px_ha if OLD is not None else np.nan, found_ha=tp * px_ha,
               missed_ha=fn * px_ha, false_alarm_ha=fp * px_ha, mapped_ha=(tp + fp) * px_ha,
               iou=tp / (tp + fp + fn) if tp + fp + fn else np.nan, precision=tp / (tp + fp) if tp + fp else np.nan,
               recall=tp / (tp + fn) if tp + fn else np.nan, labelled_px=int(gt.sum()))
    zoom = int(gt.sum()) >= SCENE_MIN_PX
    nrow = 2 if zoom else 1
    top, gap, bottom = 0.5, 0.95, 0.75
    height = (PANEL + gap) * nrow + top + bottom - gap + 0.1
    fig, axes = plt.subplots(nrow, 4, figsize=(PANEL * 4, height), squeeze=False)
    a = (height - top - bottom - (nrow - 1) * gap) / nrow
    fig.subplots_adjust(left=0.005, right=0.995, top=1 - top / height, bottom=bottom / height, wspace=0.05,
                        hspace=gap / a if nrow > 1 else 0)
    for ax in axes.flat:
        ax.set_xticks([]), ax.set_yticks([])
        for sp in ax.spines.values():
            sp.set_visible(False)

    refl = np.asarray(s["refl"][:, list(rgb)], np.float32) / spec.scale
    r, c, h, w, step = report._mosaic(idx, win, SIZE, None, MAXPX)
    img = stretch(report._paste(np.full((h, w, 3), np.nan, np.float32), refl.transpose(0, 2, 3, 1), r, c, step))
    lab = report._paste(np.full((h, w), -1, np.int16), y, r, c, step)
    pm = report._paste(np.full((h, w), -1, np.int16), pred.astype(np.int16), r, c, step)
    gaps = (lab < 0) | np.isnan(img).any(-1)
    panels(axes[0], img, lab, pm, gaps)
    axes[0, 0].set_title(f"{str(sc)[:34]}\n{role_} · {sensor} · {g:g} m", fontsize=9, loc="left")
    axes[0, 1].set_title("ground truth", fontsize=9)
    axes[0, 2].set_title(f"{NAME}: classes", fontsize=9)
    axes[0, 3].set_title(f"{NAME}: {spec.target_name} burn", fontsize=9)
    axes[0, 1].set_xlabel(f"fresh burn {ha(row['labelled_ha'])} ha · old {ha(row['old_ha'])} ha", fontsize=9)
    axes[0, 3].set_xlabel((f"IoU {f2(row['iou'])} · P {f2(row['precision'])} · R {f2(row['recall'])}\n"
                           f"found {ha(row['found_ha'])} · missed {ha(row['missed_ha'])} · false alarm "
                           f"{ha(row['false_alarm_ha'])} ha") if row["labelled_px"] else
                          f"no labelled {spec.target_name} burn · false alarm {ha(row['false_alarm_ha'])} ha", fontsize=9)
    if zoom:
        rr, cc = np.nonzero(gt)
        side = max(rr.max() - rr.min(), cc.max() - cc.min(), ZOOM_MIN) * 1.3
        cy, cx = (rr.max() + rr.min()) / 2, (cc.max() + cc.min()) / 2
        R0, C0 = int(max(0, cy - side / 2)), int(max(0, cx - side / 2))
        R1, C1 = int(min(h1, cy + side / 2)), int(min(w1, cx + side / 2))
        z = max(1, math.ceil(max(R1 - R0, C1 - C0) / ZOOM_MAX))
        near = np.nonzero((r1 < R1) & (r1 + SIZE > R0) & (c1 < C1) & (c1 + SIZE > C0))[0]
        box = lambda tiles, fill, dtype: paste_box(np.full((R1 - R0, C1 - C0) + tiles.shape[3:], fill, dtype), tiles,  # noqa: E731
                                                   r1[near], c1[near], R0, C0)[::z, ::z]
        zimg = stretch(box(refl[near].transpose(0, 2, 3, 1), np.nan, np.float32))
        zlab = box(y[near], -1, np.int16)
        zpm = box(pred[near].astype(np.int16), -1, np.int16)
        panels(axes[1], zimg, zlab, zpm, (zlab < 0) | np.isnan(zimg).any(-1))
        axes[1, 0].set_title(f"zoom: {(C1 - C0) * g / 1000:.1f} × {(R1 - R0) * g / 1000:.1f} km", fontsize=9, loc="left")
    h_ = [Patch(color=report.CLASS_COLOURS.get(n, "#999999"), label=n) for n in NAMES]
    h_ += [Patch(color=report.HIT, label=f"{spec.target_name}: found"), Patch(color=report.MISS, label="missed"),
           Patch(color=report.FALSE_ALARM, label="false alarm")]
    fig.legend(handles=h_, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=len(h_), fontsize=9, frameon=False)
    return row, fig

In [ ]:
# [5] the pages
rows, current = [], None
for n, (set_name, sensor, sc, role_) in enumerate(PICK):
    if set_name != current:
        s = D.open_set(spec, set_name, teacher=False)
        win = s["windows"].reset_index(drop=True)
        win["scene"] = win.scene.astype(str)
        current = set_name
    sel = win[win.scene == sc]
    for g in sorted(sel.gsd.unique()):
        idx = sel.index[sel.gsd == g].to_numpy()
        sub = dict(refl=np.asarray(s["refl"][idx]), mask=np.asarray(s["mask"][idx]),
                   windows=win.loc[idx].reset_index(drop=True))
        pred = report.predict_set(models, spec, sub, dev, input_mask="image")[NAME]
        row, fig = scene_page(sub, pred, sc, g, role_, sensor)
        rows.append(row)
        f = OUT / f"{n:02d}_{sensor.split()[0]}_{role_}_{sc[:31]}_{g:g}m.jpg"
        fig.savefig(f, dpi=130, pil_kwargs={"quality": 88}); plt.close(fig)
        display(Image(filename=str(f)))
        del sub, pred; gc.collect()
R = pd.DataFrame(rows)
R.to_csv(OUT / "scenes.csv", index=False)

In [ ]:
# [6] the numbers of every scene shown
cols = ["sensor", "role", "scene", "gsd", "iou", "precision", "recall", "labelled_ha", "found_ha", "missed_ha",
        "false_alarm_ha", "mapped_ha"]
R[cols].assign(scene=R.scene.str[:40]).round(3)